# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ravindidhananjana/Internship-ML/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

Ranked Action Framework & Reason Codes:We translate model score outputs into prioritized, human-reviewable content refresh queues using four distinct action archetypes:

URGENT_REFRESH (Action Score 80–100): Severe impression loss combined with position degradation on high-value pages.
Reason Codes: ERR_IMP_SLUMP, ERR_POS_DROP

MONITOR_VOLATILITY (Action Score 50–79): High rank variance without massive traffic drops yet; candidate for content stability check.
Reason Codes: ERR_VOLATILE

PRUNE_OR_CONSOLIDATE (Action Score 30–49): Low base traffic ($<20$ impressions) undergoing traffic decay; low ROI for individual rewrite.
Reason Codes: LOW_VOLUME_DECAY

NO_ACTION (Action Score $<30$): Stable search performance.
Reason Codes: OK_STABLE

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os, getpass
import duckdb
import pandas as pd
import numpy as np

# 1. Setup HF Token and DuckDB Connection
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
if not HF_TOKEN:
    HF_TOKEN = getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'fact_daily': f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_query_90d': f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

# 2. Extract Features (March 2026)
df = con.sql(f"""
    WITH windowed AS (
        SELECT
            f.client_hash_id,
            f.content_hash_id,
            SUM(CASE WHEN f.report_date > DATE '2026-03-15' THEN f.gsc_impressions ELSE 0 END) AS imp_last15,
            SUM(CASE WHEN f.report_date <= DATE '2026-03-15' THEN f.gsc_impressions ELSE 0 END) AS imp_prev15,
            AVG(CASE WHEN f.report_date <= DATE '2026-03-15' THEN f.gsc_avg_position END) AS pos_avg_prev,
            AVG(CASE WHEN f.report_date > DATE '2026-03-15' THEN f.gsc_avg_position END) AS pos_avg_curr,
            STDDEV_SAMP(CASE WHEN f.report_date <= DATE '2026-03-15' THEN f.gsc_avg_position END) AS pos_std_prev
        FROM {TABLES['fact_daily']} f
        WHERE f.report_date >= '2026-03-01' AND f.report_date <= '2026-03-31'
        GROUP BY 1, 2
        HAVING imp_prev15 >= 10
    )
    SELECT * FROM windowed
""").df().fillna({'pos_std_prev': 0, 'pos_avg_curr': 0})

# 3. Action Mapping Engine
def assign_action_playbook(row):
    reasons = []

    # Flags
    is_slump = row['imp_prev15'] > 0 and (row['imp_last15'] / row['imp_prev15']) < 0.8
    is_pos_drop = row['pos_avg_curr'] > 0 and (row['pos_avg_curr'] - row['pos_avg_prev']) >= 2.0
    is_volatile = row['pos_std_prev'] > 3.0

    if is_slump: reasons.append('ERR_IMP_SLUMP')
    if is_pos_drop: reasons.append('ERR_POS_DROP')
    if is_volatile: reasons.append('ERR_VOLATILE')

    # Priority Archetype
    if is_slump and is_pos_drop and row['imp_prev15'] >= 100:
        action = 'URGENT_REFRESH'
    elif is_volatile:
        action = 'MONITOR_VOLATILITY'
    elif is_slump and row['imp_prev15'] < 50:
        action = 'PRUNE_OR_CONSOLIDATE'
    else:
        action = 'NO_ACTION'
        if not reasons: reasons.append('OK_STABLE')

    return action, "|".join(reasons)

res = df.apply(assign_action_playbook, axis=1)
df['recommended_action'] = [r[0] for r in res]
df['reason_codes'] = [r[1] for r in res]

print("--- Action Playbook Queue Summary ---")
print(df['recommended_action'].value_counts())

Paste your Hugging Face READ token (hf_...): ··········


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

--- Action Playbook Queue Summary ---
recommended_action
MONITOR_VOLATILITY      70349
NO_ACTION               39948
URGENT_REFRESH           7541
PRUNE_OR_CONSOLIDATE     2675
Name: count, dtype: int64


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

Intended Scope & Bounds:

Intended Use: Serves as a decision-support queue for SEO strategists and editorial teams to prioritize manual content audits and refreshes across portfolio assets.

Operational Boundaries:

Non-Causal: High risk scores flag observational rank and impression correlations, not guaranteed causal decline.

Domain Boundaries: Model thresholds are calibrated for organic Google Search performance metrics (GSC impressions/rankings); they do not model direct paid traffic or social media campaigns.

Time Boundaries: Recommendations expire after 14 days due to search index shifting.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

Human Review Rules:

Sanity Check Intention: Verify whether impression drops are driven by planned URL migrations, seasonal intent shifts, or SERP layout changes (e.g., insertion of new AI Overviews or ad blocks).

Context Verification: Ensure low-volume pages are not critical brand assets (e.g., contact/terms pages).

The No-Go Automation List (NEVER Automate):

Automatic AI Content Overwrites: Never automatically replace live web page content using automated LLM rewrites without human editorial sign-off.

Automatic URL Deletions / Redirects: Never automate page pruning or 301 redirects programmatically without human SEO review.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

Retrain & Drift Triggers:Concept Drift Trigger: If portfolio-wide baseline impression slump rates shift by $>15\%$ month-over-month (indicating a major core search algorithm update).

Precision Degradation: If human reviewer agreement rate on URGENT_REFRESH recommendations drops below $70\%$.

Cadence Trigger: Retrain models bi-weekly on updated 15-day rolling performance windows.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Save Action Queue CSV to work/outputs/
os.makedirs('work/outputs', exist_ok=True)
export_cols = ['client_hash_id', 'content_hash_id', 'recommended_action', 'reason_codes', 'imp_prev15', 'imp_last15', 'pos_avg_prev']

df_ranked = df.sort_values(by=['recommended_action', 'imp_prev15'], ascending=[True, False])
df_ranked[export_cols].to_csv('work/outputs/action_playbook_queue.csv', index=False)

print(f"Action Playbook successfully exported to work/outputs/action_playbook_queue.csv ({len(df_ranked)} rows)")

Action Playbook successfully exported to work/outputs/action_playbook_queue.csv (120513 rows)


## Self-check

Before you submit, confirm each line honestly:

- [x ] Every section above is filled — markdown thinking AND the code that backs it
- [x ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x ] No client names, URLs, or private queries anywhere
- [x ] My claims use careful words: observed, measured, directional, decision-support
- [x ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.